
- **Exercício 3.1** – Comparar `SelfAttention_v1` e `SelfAttention_v2`
- **Exercício 3.2** – Context vectors com 2 dimensões
- **Exercício 3.3** – Módulo de attention do tamanho do GPT-2

In [ ]:
import torch
import torch.nn as nn

print("torch:", torch.__version__)

inputs = torch.tensor(
    [[0.43, 0.15, 0.89],  # Your     (x^1)
     [0.55, 0.87, 0.66],  # journey  (x^2)
     [0.57, 0.85, 0.64],  # starts   (x^3)
     [0.22, 0.58, 0.33],  # with     (x^4)
     [0.77, 0.25, 0.10],  # one      (x^5)
     [0.05, 0.80, 0.55]]  # step     (x^6)
)
d_in = inputs.shape[1]  # 3
d_out = 2

torch: 2.14.1+cpu


## Exercício 3.1 – Comparar `SelfAttention_v1` e `SelfAttention_v2`

As duas classes (listings 3.1 e 3.2) dão resultados diferentes porque inicializam os pesos de jeitos diferentes. A tarefa é copiar os pesos de uma instância da v2 para uma da v1 para que as saídas fiquem iguais.

In [2]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec


class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

Antes da transferência, as saídas são diferentes:

In [3]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)

print("v1:\n", sa_v1(inputs))
print("v2:\n", sa_v2(inputs))

v1:
 tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)
v2:
 tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


Formatos dos pesos: na v1 é `(d_in, d_out)`; no `nn.Linear` é `(d_out, d_in)`.

In [4]:
print("v1 W_query:", sa_v1.W_query.shape)
print("v2 W_query.weight:", sa_v2.W_query.weight.shape)

v1 W_query: torch.Size([3, 2])
v2 W_query.weight: torch.Size([2, 3])


Transferindo os pesos da v2 para a v1 (com transposição):

In [5]:
sa_v1.W_query = nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key = nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = nn.Parameter(sa_v2.W_value.weight.T)

out_v1 = sa_v1(inputs)
out_v2 = sa_v2(inputs)
print("v1:\n", out_v1)
print("v2:\n", out_v2)
print("Saídas iguais?", torch.allclose(out_v1, out_v2))

v1:
 tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
v2:
 tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
Saídas iguais? True


**Conclusão:** `nn.Linear` calcula `x @ W.T`, então basta passar `weight.T` para a v1. Com os mesmos pesos, as duas implementações produzem o mesmo resultado.

## Exercício 3.2 – Context vectors com 2 dimensões


In [6]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec


class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(
                d_in, d_out, context_length, dropout, qkv_bias
            )
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


batch = torch.stack((inputs, inputs), dim=0)
print("batch.shape:", batch.shape)

batch.shape: torch.Size([2, 6, 3])


In [7]:
torch.manual_seed(123)
context_length = batch.shape[1]
mha = MultiHeadAttentionWrapper(
    d_in, d_out=2, context_length=context_length, dropout=0.0, num_heads=2)
context_vecs = mha(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 4])


**Solução:** mantendo `num_heads=2`, basta usar `d_out=1`. Cada head devolve 1 dimensão e a concatenação resulta em 2.

In [8]:
torch.manual_seed(123)
mha = MultiHeadAttentionWrapper(
    d_in, d_out=1, context_length=context_length, dropout=0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]],

        [[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


## Exercício 3.3 – Módulo de attention do tamanho do GPT-2

Com a classe `MultiHeadAttention` (listing 3.5), crie um módulo com a configuração do menor GPT-2: 12 heads, embeddings de 768 dimensões e context length de 1.024.

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out,
                 context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )
        context_vec = self.out_proj(context_vec)
        return context_vec

**Solução:**

In [10]:
context_length = 1024
d_in, d_out = 768, 768
num_heads = 12

mha_gpt2 = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads)

print("head_dim:", mha_gpt2.head_dim)
print("Parâmetros:", sum(p.numel() for p in mha_gpt2.parameters()))

head_dim: 64
Parâmetros: 2360064


Testando com um batch aleatório de 2 sequências de 1.024 tokens:

In [11]:
torch.manual_seed(123)
x = torch.randn(2, context_length, d_in)
out = mha_gpt2(x)
print("Entrada:", x.shape)
print("Saída:  ", out.shape)

Entrada: torch.Size([2, 1024, 768])
Saída:   torch.Size([2, 1024, 768])


**Conclusão:** cada uma das 12 heads trabalha com `768 / 12 = 64` dimensões. O módulo tem cerca de 2,36 milhões de parâmetros (4 matrizes 768×768 mais o bias de `out_proj`) e mantém o formato `(batch, tokens, 768)` na saída.